# Аналитические витрины


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

postgres_url = f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'admin')}:{os.getenv('POSTGRES_PASSWORD', 'admin')}@{os.getenv('POSTGRES_HOST', 'postgres')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'oil_analytics')}"
engine = create_engine(postgres_url)


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text


def clean_production(frame):
    result = frame.copy()
    result["oil_ton"] = result["oil_ton"].fillna(result["oil_ton"].median())
    result["downtime_hours"] = result["downtime_hours"].fillna(0)
    result["pressure"] = result.groupby("well_id")["pressure"].transform(lambda x: x.fillna(x.median()))
    result["temperature"] = result.groupby("well_id")["temperature"].transform(lambda x: x.fillna(x.median()))
    z = (result["oil_ton"] - result["oil_ton"].mean()) / result["oil_ton"].std(ddof=0)
    return result[z.abs().fillna(0) <= 3]



def daily_telemetry(frame):
    data = frame.copy().sort_values(["well_id", "timestamp"])
    data["date"] = data["timestamp"].dt.normalize()
    columns = ["pump_speed_rpm", "pump_current", "pressure_in", "pressure_out", "temperature", "vibration", "oil_flow_rate"]
    for column in columns:
        data[column] = data.groupby("well_id")[column].transform(lambda x: x.ffill().bfill().fillna(x.median()))
    result = data.groupby(["well_id", "date"], as_index=False).agg(
        avg_pressure_in=("pressure_in", "mean"),
        avg_pressure_out=("pressure_out", "mean"),
        avg_temperature=("temperature", "mean"),
        avg_pump_current=("pump_current", "mean"),
        avg_pump_speed_rpm=("pump_speed_rpm", "mean"),
        avg_vibration=("vibration", "mean"),
        avg_oil_flow_rate=("oil_flow_rate", "mean"),
        pump_runtime_hours=("record_id", "count"),
    )
    result["avg_pressure"] = (result["avg_pressure_in"] + result["avg_pressure_out"]) / 2
    return result


In [3]:
wells = pd.read_sql("SELECT * FROM wells", engine)
production = pd.read_sql("SELECT * FROM production", engine, parse_dates=["date"])
telemetry = pd.read_sql("SELECT * FROM well_telemetry", engine, parse_dates=["timestamp"])
deliveries = pd.read_sql("SELECT * FROM deliveries", engine, parse_dates=["date"])
drivers = pd.read_sql("SELECT * FROM drivers", engine)
production = clean_production(production)
telemetry_daily = daily_telemetry(telemetry)
fact = production.merge(telemetry_daily, on=["well_id", "date"], how="left")
fact["avg_pressure"] = fact["avg_pressure"].fillna(fact["pressure"])
fact["avg_temperature"] = fact["avg_temperature"].fillna(fact["temperature"])
fact = fact.merge(wells[["well_id", "name", "field_name", "region", "status"]], on="well_id", how="left")
fact["downtime_ratio"] = (fact["downtime_hours"] / 24).clip(0, 1)
daily_production = fact.groupby("date", as_index=False).agg(total_oil_ton=("oil_ton", "sum"), avg_pressure=("avg_pressure", "mean"), avg_temperature=("avg_temperature", "mean"), downtime_ratio=("downtime_ratio", "mean"))
well_kpi = fact.groupby(["well_id", "name", "field_name", "region", "status"], as_index=False).agg(avg_flow_tpd=("oil_ton", "mean"), total_oil_ton=("oil_ton", "sum"), downtime_pct=("downtime_ratio", lambda x: x.mean() * 100), avg_pressure=("avg_pressure", "mean"), avg_temperature=("avg_temperature", "mean")).sort_values("avg_flow_tpd", ascending=False)
well_kpi["rank_by_flow"] = range(1, len(well_kpi) + 1)
temp_pressure_effects = fact[["date", "well_id", "name", "oil_ton", "avg_pressure", "avg_temperature", "energy_kwh", "downtime_ratio"]].copy()
daily_production.head(), well_kpi.head()


(        date  total_oil_ton  avg_pressure  avg_temperature  downtime_ratio
 0 2025-10-01          717.5    109.488542        84.634375        0.230833
 1 2025-10-02          717.3    115.475000        84.625000        0.233333
 2 2025-10-03          719.2    115.500000        84.875000        0.231667
 3 2025-10-04          722.9    116.000000        84.375000        0.222500
 4 2025-10-05          721.0    115.725000        84.675000        0.229167,
    well_id      name    field_name         region       status  avg_flow_tpd  \
 0        1  Well-101   Severo-Ural   Khanty-Mansi       active    213.150000   
 4        5  Well-305   Zapad-Field          Tomsk       active    198.430000   
 1        2  Well-102   Severo-Ural   Khanty-Mansi       active    185.816667   
 2        3  Well-203  Vostok-Field  Yamalo-Nenets  maintenance    121.696667   
 3        4  Well-304   Zapad-Field          Tomsk    suspended      0.000000   
 
    total_oil_ton  downtime_pct  avg_pressure  avg_temp

In [4]:
deliveries["cost_per_km"] = deliveries["cost_usd"] / deliveries["distance_km"]
joined = deliveries.merge(drivers, on="driver_id", how="left", suffixes=("", "_driver"))


In [5]:
if "delay_rate" not in joined:
    joined["delay_rate"] = (joined["delay_hours"] > 0).astype(int)
if "route_efficiency_score" not in joined:
    joined["route_efficiency_score"] = joined["volume_ton"] / (1 + joined["cost_per_km"] + joined["delay_hours"])
delivery_kpi = joined.groupby(["driver_id", "name", "weather_conditions"], as_index=False).agg(avg_delay_hours=("delay_hours", "mean"), avg_cost_per_km=("cost_per_km", "mean"), total_volume_ton=("volume_ton", "sum"), delivery_count=("delivery_id", "count"), avg_distance_km=("distance_km", "mean"), delay_rate=("delay_rate", "mean"), route_efficiency_score=("route_efficiency_score", "mean"))
delivery_kpi.head()


,driver_id,name,weather_conditions,avg_delay_hours,avg_cost_per_km,total_volume_ton,delivery_count,avg_distance_km,delay_rate,route_efficiency_score
0,1,Ivan Petrov,Clear,0.00,11.696993,134.2,4,187.75,0.0,2.643198
1,1,Ivan Petrov,Rain,1.00,11.846316,34.2,1,190.00,1.0,2.469971
2,2,Sergey Sidorov,Clear,0.50,12.933456,56.3,2,147.50,0.5,1.961753
3,2,Sergey Sidorov,Cloudy,0.25,11.973602,53.5,2,149.50,0.5,2.022930
4,2,Sergey Sidorov,Fog,1.25,11.974103,57.4,2,154.50,1.0,2.017956


In [6]:
with engine.begin() as connection:
    for table in ["mart_daily_production", "mart_well_kpi", "mart_temp_pressure_effects", "mart_delivery_kpi"]:
        connection.execute(text(f"DROP TABLE IF EXISTS {table}"))
daily_production.to_sql("mart_daily_production", engine, index=False, if_exists="replace")
well_kpi.to_sql("mart_well_kpi", engine, index=False, if_exists="replace")
temp_pressure_effects.to_sql("mart_temp_pressure_effects", engine, index=False, if_exists="replace")
delivery_kpi.to_sql("mart_delivery_kpi", engine, index=False, if_exists="replace")
pd.read_sql("SELECT 'mart_daily_production' AS table_name, COUNT(*) AS rows_count FROM mart_daily_production UNION ALL SELECT 'mart_well_kpi', COUNT(*) FROM mart_well_kpi UNION ALL SELECT 'mart_temp_pressure_effects', COUNT(*) FROM mart_temp_pressure_effects UNION ALL SELECT 'mart_delivery_kpi', COUNT(*) FROM mart_delivery_kpi", engine)


,table_name,rows_count
0,mart_daily_production,30
1,mart_well_kpi,5
2,mart_temp_pressure_effects,150
3,mart_delivery_kpi,13
